# Task 2.8 — Tiền xử lý, embedding và TabNet

**Inputs:** competition data + outputs **2.6 và 2.7**. Chạy trên Kaggle GPU.
Các thay đổi dùng cùng holdout seed 9. Cấu hình MLP tốt nhất 2.7 làm gốc; mỗi biến thể chỉ thay một nhóm.
Không suy luận đặc trưng từ test, không đọc leaderboard. TabNet chạy sau các biến thể MLP/embedding.


In [ ]:
from pathlib import Path
import importlib.util
import sys
import hashlib
import json

DATA_DIR = None  # Auto-detect inputs; set a directory here only to disambiguate.

def resolve_data_dir(data_dir=None, expected_sha256=None,
                     required_files=("train.csv",), input_root="/kaggle/input"):
    """Find attached data without assuming Kaggle's mount layout."""
    from pathlib import Path
    import hashlib

    root = Path(input_root)
    if data_dir is not None:
        candidates = [Path(data_dir)]
        missing = [name for name in required_files if not (candidates[0] / name).is_file()]
        if missing:
            raise FileNotFoundError(
                f"DATA_DIR={data_dir} is missing {missing}. "
                "Attach competition data or set DATA_DIR=None for automatic discovery."
            )
    else:
        candidates = sorted({
            path.parent for path in root.rglob("train.csv")
            if path.is_file() and all((path.parent / name).is_file() for name in required_files)
        })
        if not candidates:
            mounted = sorted(path.name for path in root.iterdir()) if root.is_dir() else []
            raise FileNotFoundError(
                f"No attached directory contains {list(required_files)} under {root}. "
                "Use Add Input to attach competition playground-series-s6e11. "
                f"Mounted entries: {mounted}"
            )

    if expected_sha256 is not None:
        matching = []
        for directory in candidates:
            digest = hashlib.sha256()
            with (directory / "train.csv").open("rb") as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    digest.update(block)
            if digest.hexdigest() == expected_sha256:
                matching.append(directory)
        if not matching:
            raise ValueError(
                "Attached train.csv does not match the task 2.6 checksum. "
                f"Check the competition data/version. Candidates: {candidates}"
            )
        candidates = matching
    elif len(candidates) > 1:
        bundles = [directory for directory in candidates
                   if all((directory / name).is_file()
                          for name in ("train.csv", "test.csv", "sample_submission.csv"))]
        if bundles:
            candidates = bundles

    if len(candidates) != 1:
        raise ValueError(
            f"Multiple valid data directories found: {candidates}. "
            "Set DATA_DIR to the exact directory you want to use."
        )
    return candidates[0]

# Nếu gắn nhiều version 2.6, điền chính xác thư mục version cần dùng.
FOUNDATION_DIR = None
if FOUNDATION_DIR is None:
    found = sorted(Path("/kaggle/input").rglob("foundation_manifest.json"))
    if len(found) != 1:
        raise FileNotFoundError(f"Attach exactly one task 2.6 output; found: {found}")
    FOUNDATION_DIR = found[0].parent
else:
    FOUNDATION_DIR = Path(FOUNDATION_DIR)
manifest = json.loads((FOUNDATION_DIR / "foundation_manifest.json").read_text(encoding="utf-8"))
engine_file = FOUNDATION_DIR / "credit_fraud_dl_engine.py"
if hashlib.sha256(engine_file.read_bytes()).hexdigest() != manifest["engine_sha256"]:
    raise ValueError("Foundation engine checksum mismatch.")
spec = importlib.util.spec_from_file_location("credit_fraud_dl_engine", engine_file)
dl = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = dl
spec.loader.exec_module(dl)

import numpy as np
import pandas as pd
DATA_DIR = resolve_data_dir(DATA_DIR, expected_sha256=manifest["train_sha256"])
print("Resolved DATA_DIR:", DATA_DIR)
train_path = DATA_DIR / "train.csv"
dl.verify_source(train_path, manifest)
train = pd.read_csv(train_path)
dl.validate_train(train)
for name, digest in manifest["membership_sha256"].items():
    if dl.file_hash(FOUNDATION_DIR / name) != digest:
        raise ValueError(f"Foundation membership file changed: {name}")
split = pd.read_csv(FOUNDATION_DIR / "holdout_split_seed9.csv")
train_idx, valid_idx = dl.holdout_indices(train, split)
print("Foundation:", manifest["foundation_id"])
print("Runtime:", dl.environment())


## 2.8.1 — Nạp quyết định tuning và ghi bảng thí nghiệm

Bảng tuning 2.7 có baseline làm control. Kiểm tra foundation ID trước khi dùng config. Lưu cấu hình và metric của mỗi biến thể.


In [ ]:
OUTPUT_DIR = Path("/kaggle/working/task_2_8_dl_features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TABNET = True  # Cần pytorch-tabnet có sẵn hoặc bật Internet cho pip.
tuning = dl.json_read(dl.find_input("tuning_manifest.json"))
if tuning["foundation_id"] != manifest["foundation_id"]:
    raise ValueError("Tuning and feature experiments use different foundations.")
base_cfg = tuning["best_config"]
results = {r["config_id"]: r for r in tuning["results"]}

def save_features():
    ordered = dl.ranked(list(results.values()))
    dl.json_write(OUTPUT_DIR / "feature_manifest.json", {
        "foundation_id": manifest["foundation_id"], "source_sha256": manifest["train_sha256"],
        "results": ordered, "best_config": ordered[0]["config"],
        "tabnet_enabled": RUN_TABNET,
        "log_columns_selected_from_training_only": log_columns,
        "selection_limit": "Holdout screening; not an unbiased final score.",
    })
    table = pd.DataFrame([{
        "config_id": r["config_id"], "family": r["config"]["family"],
        "roc_auc": r["roc_auc"], "average_precision": r["average_precision"],
        "best_epoch": r["best_epoch"],
    } for r in ordered])
    table.to_csv(OUTPUT_DIR / "feature_experiments.csv", index=False)
    return table

def run_variant(cfg):
    key = dl.config_id(cfg)
    if key not in results:
        results[key] = dl.train_run(
            train, train_idx, valid_idx, cfg, OUTPUT_DIR / key, manifest["train_sha256"])
        save_features()
    return results[key]


## 2.8.2 — Log cho cột lệch phải, chu kỳ giờ/ngày và Boolean

Chỉ xem phân phối phần train của holdout. log1p chỉ áp dụng cột có giá trị không âm và skew > 1; nếu không có, ghi nhận không áp dụng.
Chu kỳ giữ cột gốc và bổ sung sin/cos. Mỗi nhánh là một thí nghiệm riêng; sau đó mới thử kết hợp log + chu kỳ.
Median/scaler của mỗi nhánh vẫn chỉ fit trên train split.


In [ ]:
log_columns = []
for col in ["amount_usd", "account_balance_usd", "distance_from_home_km", "hours_since_last_txn"]:
    if col not in train:
        continue
    values = pd.to_numeric(train.iloc[train_idx][col], errors="coerce").replace(
        [np.inf, -np.inf], np.nan).dropna()
    if len(values) and values.min() >= 0 and values.skew() > 1:
        log_columns.append(col)
if log_columns:
    run_variant(dl.config(**{**base_cfg, "log_columns": log_columns}))
run_variant(dl.config(**{**base_cfg, "cyclical": True}))
run_variant(dl.config(**{**base_cfg, "scale_boolean": False}))
if log_columns:
    run_variant(dl.config(**{**base_cfg, "log_columns": log_columns, "cyclical": True}))
print("Log candidates selected from train only:", log_columns)
display(save_features())


## 2.8.3 — Embedding MLP

Thay riêng biểu diễn categorical: kích thước tự động hoặc 4 chiều/category. Các tham số mạng và optimizer kế thừa ứng viên MLP, không thêm mô hình ML cổ điển.


In [ ]:
for embedding_dim in [None, 4]:
    run_variant(dl.config(**{**base_cfg, "family": "embedding", "embedding_dim": embedding_dim}))
display(save_features())


## 2.8.4 — TabNet: baseline và 3 cấu hình giới hạn

TabNet dùng Adam, không class weight; thay attention width, số decision steps, embedding và LR.
4 cấu hình tối đa, early stopping theo AUC. Đây là thử nghiệm một số cấu hình, không tìm kiếm toàn bộ không gian.
Nếu bỏ qua TabNet vì compute, đặt RUN_TABNET=False trước Run All; manifest ghi rõ việc này.


In [ ]:
if RUN_TABNET:
    for width, steps, learning_rate, embedding_dim in [
        (16, 4, 0.02, 1),
        (16, 4, 0.005, 3),
        (32, 4, 0.01, 3),
        (16, 6, 0.01, 3),
    ]:
        tabnet_cfg = dl.config(
            family="tabnet", learning_rate=learning_rate, weight_decay=0.0,
            max_epochs=60, patience=10, positive_weight=1.0, scheduler=False,
            tabnet={**dl.BASELINE["tabnet"], "n_d": width, "n_a": width,
                    "n_steps": steps, "cat_emb_dim": embedding_dim},
        )
        run_variant(tabnet_cfg)
display(save_features())
print("Candidates are screening results; confirm shortlisted configs in task 2.9.")
